In [ ]:
"""
04_experiment_comparison.ipynb
================================
Copy each section between # %% markers into separate notebook cells.
Update EXPERIMENTS dict with your actual experiment directories.
"""

# 04 - Experiment Comparison

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── UPDATE THESE PATHS ───
EXPERIMENTS = {
    "exp_001": "../training_logs/exp_001_yolo26l_full_finetune",
    # Add more experiments as you run them:
    # "exp_002": "../training_logs/exp_002_yolo26l_imgsz960",
    # "exp_003": "../training_logs/exp_003_yolo26m_full_finetune",
}
# ───────────────────────────

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

SAVE_DIR = os.path.join(os.path.dirname(list(EXPERIMENTS.values())[0]), "comparison_plots")
os.makedirs(SAVE_DIR, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(os.path.join(SAVE_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    print(f"Saved: {name}.png")

In [ ]:
exp_data = {}

for name, exp_dir in EXPERIMENTS.items():
    data = {"name": name, "dir": exp_dir}

    # Load results.csv
    csv_path = os.path.join(exp_dir, "metrics", "results.csv")
    if not os.path.exists(csv_path):
        csv_path = os.path.join(exp_dir, "yolo_output", "results.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        data["df"] = df

    # Load custom_metrics.json
    custom_path = os.path.join(exp_dir, "metrics", "custom_metrics.json")
    if os.path.exists(custom_path):
        with open(custom_path) as f:
            data["custom"] = json.load(f)

    # Load test results
    test_path = os.path.join(exp_dir, "metrics", "test_results.json")
    if os.path.exists(test_path):
        with open(test_path) as f:
            data["test"] = json.load(f)

    exp_data[name] = data
    print(f"Loaded {name}: {len(data.get('df', []))} epochs")

In [ ]:
print("=" * 80)
print("  EXPERIMENT COMPARISON")
print("=" * 80)

headers = f"{'Experiment':<20} {'mAP@50':>8} {'mAP50-95':>10} {'Best Ep':>8} {'Time (min)':>10}"
print(headers)
print("-" * 60)

for name, data in exp_data.items():
    df = data.get("df")
    custom = data.get("custom", {})

    map50_col = [c for c in df.columns if "mAP50" in c and "mAP50-95" not in c] if df is not None else []
    map5095_col = [c for c in df.columns if "mAP50-95" in c] if df is not None else []

    best_map50 = f"{df[map50_col[0]].max():.4f}" if map50_col else "N/A"
    best_map5095 = f"{df[map5095_col[0]].max():.4f}" if map5095_col else "N/A"
    best_epoch = str(custom.get("best_epoch", "N/A"))
    train_time = f"{custom.get('training_time_seconds', 0)/60:.1f}" if custom else "N/A"

    print(f"{name:<20} {best_map50:>8} {best_map5095:>10} {best_epoch:>8} {train_time:>10}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors_list = plt.cm.tab10.colors
for i, (name, data) in enumerate(exp_data.items()):
    df = data.get("df")
    if df is None:
        continue

    map50_col = [c for c in df.columns if "mAP50" in c and "mAP50-95" not in c]
    if map50_col:
        ax.plot(df.index, df[map50_col[0]], label=name,
                color=colors_list[i % len(colors_list)], linewidth=2)

        # Mark best
        best_idx = df[map50_col[0]].idxmax()
        best_val = df[map50_col[0]].max()
        ax.scatter(best_idx, best_val, color=colors_list[i % len(colors_list)],
                   s=100, zorder=5, edgecolors="black")

ax.set_xlabel("Epoch")
ax.set_ylabel("mAP@50")
ax.set_title("mAP@50 Convergence — All Experiments")
ax.legend()
fig.tight_layout()
save_fig(fig, "01_map50_comparison")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for i, (name, data) in enumerate(exp_data.items()):
    df = data.get("df")
    if df is None:
        continue

    # Sum all training losses
    loss_cols = [c for c in df.columns if "train" in c and "loss" in c]
    if loss_cols:
        total_loss = df[loss_cols].sum(axis=1)
        ax.plot(df.index, total_loss, label=f"{name} (train)",
                color=colors_list[i % len(colors_list)], linewidth=1.5)

    val_loss_cols = [c for c in df.columns if "val" in c and "loss" in c]
    if val_loss_cols:
        val_total = df[val_loss_cols].sum(axis=1)
        ax.plot(df.index, val_total, label=f"{name} (val)",
                color=colors_list[i % len(colors_list)], linewidth=1.5, linestyle="--")

ax.set_xlabel("Epoch")
ax.set_ylabel("Total Loss")
ax.set_title("Total Loss — All Experiments")
ax.legend()
fig.tight_layout()
save_fig(fig, "02_loss_comparison")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

class_names = ["bike", "helmet", "no-helmet", "number-plate"]
x = np.arange(len(class_names))
n_exps = len(exp_data)
width = 0.8 / max(n_exps, 1)

for i, (name, data) in enumerate(exp_data.items()):
    test = data.get("test", {})
    per_class = test.get("per_class", {})

    if per_class:
        ap_vals = [per_class.get(cls, {}).get("ap50", 0) for cls in class_names]
    else:
        # Try from custom metrics (last epoch)
        custom = data.get("custom", {})
        per_class_list = custom.get("per_class_ap", [])
        if per_class_list:
            last = per_class_list[-1]
            ap_vals = [last.get(cls, {}).get("ap50", 0) for cls in class_names]
        else:
            ap_vals = [0] * len(class_names)

    bars = ax.bar(x + i * width, ap_vals, width, label=name, alpha=0.8)

    for bar, val in zip(bars, ap_vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.2f}", ha="center", fontsize=8)

ax.set_xticks(x + width * (n_exps - 1) / 2)
ax.set_xticklabels(class_names)
ax.set_ylabel("AP@50")
ax.set_title("Per-Class AP@50 Comparison")
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
save_fig(fig, "03_per_class_comparison")
plt.show()

In [ ]:
has_gradients = any(data.get("custom", {}).get("gradients") for data in exp_data.values())

if has_gradients:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for group_idx, group in enumerate(["backbone", "neck", "head"]):
        ax = axes[group_idx]
        for i, (name, data) in enumerate(exp_data.items()):
            custom = data.get("custom", {})
            gradients = custom.get("gradients", [])
            if gradients:
                epochs = [g["epoch"] for g in gradients]
                norms = [g.get(group, {}).get("l2_norm", 0) for g in gradients]
                if any(v > 0 for v in norms):
                    ax.plot(epochs, norms, label=name,
                            color=colors_list[i % len(colors_list)], linewidth=1.5)

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Gradient L2 Norm")
        ax.set_title(f"{group.capitalize()} Gradients")
        ax.legend()

    fig.suptitle("Gradient Norm Comparison", fontsize=14)
    fig.tight_layout()
    save_fig(fig, "04_gradient_comparison")
    plt.show()
else:
    print("No gradient data available for comparison")

In [ ]:
print("\n" + "=" * 80)
print("  FINAL RECOMMENDATION")
print("=" * 80)

best_exp = None
best_map = 0

for name, data in exp_data.items():
    df = data.get("df")
    if df is None:
        continue
    map50_col = [c for c in df.columns if "mAP50" in c and "mAP50-95" not in c]
    if map50_col:
        val = df[map50_col[0]].max()
        if val > best_map:
            best_map = val
            best_exp = name

if best_exp:
    print(f"\n  Best experiment: {best_exp} (mAP@50 = {best_map:.4f})")
    print(f"  Use weights from: {EXPERIMENTS[best_exp]}/weights/best.pt")